- [Deltakit stim](https://github.com/Deltakit/deltakit-stim) is a fork of stim which adds the functionality to simulate leakage. It has four additional [operations](https://github.com/Deltakit/deltakit-stim/blob/main/doc/deltakit_stim_gates.md):
  - **LEAKAGE**
    - depolarising leakage
    - any future qubits this qubit interacts with also get fully depolarised
    - *(Future work: could look into the docs to make a less harmful leakage model, like Mølmer–Sørensen leakage as per Brown's papers)*
    - reset gates return leaked qubits to the computational subspace
  - **RELAX**
    - relaxes a leaked qubit with a given probability; otherwise does nothing
  - **HERALD_LEAKAGE_EVENT**
    - is a leakage event but also populates the measurement record with this data
    - question: I have a leakage-detecting measurement at the end of each round. It can tell me if a qubit is leaked but not exactly where. Wonder if I can somehow use `HERALD_LEAKAGE_EVENT` in combination with this
  - **RL**
    - resets target qubit into the computational subspace

<br>

- See some circuit examples at:
  - https://deltakit-docs.riverlane.com/en/stable/guide/simulation.html#leakage-simulation

In [ ]:
import sys
import os
import deltakit_stim

# Save deltakit_stim as stim so any other calls to import stim (including from other packages) just use deltakit_stim:
injections = ['stim._detect_machine_architecture',
'stim._stim_polyfill',
'stim']
for namespace in injections:
    sys.modules[namespace] = sys.modules[f"deltakit_{namespace}"]

sys.path.append(os.path.abspath("src"))
from bb_ions import *

In [ ]:
circuit = stim.Circuit("""
R 0 1
H 0 
LEAKAGE(0.1) 0 
CX 0 1
TICK
HERALD_LEAKAGE_EVENT() 0 1
M 0 1
""")
display(circuit.diagram("timeline-svg"))
sampler_object = circuit.compile_sampler()
print(sampler_object.sample(shots = 10).astype(int))




In [ ]:
circuit = stim.Circuit("""R 0 1
TICK
X_ERROR(0.02) 0 1
LEAKAGE(0.005) 0 1
H 0
TICK
DEPOLARIZE1(0.001) 0
RELAX(0.002) 0 1
X 1
TICK
DEPOLARIZE1(0.001) 1
RELAX(0.002) 1 0
CX 0 1
TICK
DEPOLARIZE2(0.01) 0 1
LEAKAGE(0.005) 0 1
RELAX(0.005) 0 1
H 0
TICK
DEPOLARIZE1(0.001) 0
RELAX(0.002) 0 1
I 1
TICK
DEPOLARIZE1(0.001) 1
RELAX(0.002) 1 0
HERALD_LEAKAGE_EVENT(0.05) 0 1
M(0.05) 0 1
DETECTOR rec[-4]
DETECTOR rec[-3]
""")

circuit.diagram("timeline-svg")